In [8]:

import yfinance as yf
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.preprocessing import StandardScaler
# from matplotlib.ticker import MultipleLocator
# import re
# from tqdm import tqdm
# import scipy

# Adquiriendo Datos

In [9]:
#### Abriendo lista de top 40 en SPLAC
with open('SPLAC_Companias.txt', 'r', encoding='utf-8') as file:
    tickers = file.readlines()

tickers = [elem.rstrip('\n') for elem in tickers]
    #Borrando newline
tickers = sorted(tickers)
    # Ordenando alfabeticamente

In [10]:
# ##### Descarga inicial de datos
# start_date = "2003-01-01"
# end_date = "2026-06-15"
# df_crudo = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)
#     # Al descargar, yf automaticamente los ordena alfabeticamente
# df_crudo.to_csv('datos_crudos_splac_v2.csv', index=True, encoding='utf-8')

In [11]:
##### Lectura y Limpeza de datos

#### Cargando datos
df_crudos = pd.read_csv('datos_crudos_splac_v2.csv')

#### Limpieza
### Arreglando nombres de todas las columnas
nombres = df_crudos.columns.to_list()
for i in range(1,len(nombres)):
    nombres[i] = tickers[(i-1)%len(tickers)] + '.' + nombres[i]
nombres[0] = 'Date' #Primera columna son fechas
df_crudos.columns = nombres #Aplicando los cambios

### Eliminando primera fila, redundancia
    # Primera fila es una repeticion de nombre de acciones
df_crudos = df_crudos.drop(index=0).reset_index(drop=True)
    #"reset_index": reinicia los indices

### Cambiando el index del conjunto
df_crudos = df_crudos.set_index('Date',drop=True)
    #"drop=True": elimina la columna de "Date" al cambiarlo a index

C:\Users\JP\AppData\Local\Temp\ipykernel_11420\2721882548.py:4: DtypeWarning: Columns (0: Adj Close, 1: Adj Close.1, 2: Adj Close.2, 3: Adj Close.3, 4: Adj Close.4, 5: Adj Close.5, 6: Adj Close.6, 7: Adj Close.7, 8: Adj Close.8, 9: Adj Close.9, 10: Adj Close.10, 11: Adj Close.11, 12: Adj Close.12, 13: Adj Close.13, 14: Adj Close.14, 15: Adj Close.15, 16: Adj Close.16, 17: Adj Close.17, 18: Adj Close.18, 19: Adj Close.19, 20: Adj Close.20, 21: Adj Close.21, 22: Adj Close.22, 23: Adj Close.23, 24: Adj Close.24, 25: Adj Close.25, 26: Adj Close.26, 27: Adj Close.27, 28: Adj Close.28, 29: Adj Close.29, 30: Adj Close.30, 31: Adj Close.31, 32: Adj Close.32, 33: Adj Close.33, 34: Adj Close.34, 35: Adj Close.35, 36: Adj Close.36, 37: Adj Close.37, 38: Adj Close.38, 39: Adj Close.39, 40: Close, 41: Close.1, 42: Close.2, 43: Close.3, 44: Close.4, 45: Close.5, 46: Close.6, 47: Close.7, 48: Close.8, 49: Close.9, 50: Close.10, 51: Close.11, 52: Close.12, 53: Close.13, 54: Close.14, 55: Close.15, 56:

In [12]:
##### Creando precios de datos crudos
#### Seleccionando columnas adecuadas
bool_lista = df_crudos.columns.str.contains(r'Adj Close')
    # Solo nos vamos a enfocar en 'Adj Close'
precios_nombres = []
for col_nom, bol in zip(df_crudos.columns.to_list(), bool_lista):
    if bol: precios_nombres.append(col_nom)

#### Copiando datos originales para mantener integridad
df_precios=df_crudos[precios_nombres].copy()

#### Cambiando nombre de columnas
tickers = [elem.split('.')[0] for elem in tickers]
df_precios.columns = tickers

#### Eliminando empresas sin suficiente datos
max_dias = len(df_precios)
THRESH = 0.2    #Cantidad de dias originales dispuesto a borrar, 0.2 => 20%
for col in df_precios.columns:
    if df_precios[col].isna().sum() > THRESH * max_dias:
        df_precios = df_precios.drop(col, axis=1)

#### Hacer forward fill y eliminar Nans
df_precios = df_precios.ffill().dropna()


#### Transformat tipos de datos de str a float
df_precios = df_precios.astype(float)

df_precios.info()


<class 'pandas.DataFrame'>
Index: 5292 entries, 2006-02-27 to 2026-06-12
Data columns (total 32 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ABEV       5292 non-null   float64
 1   AC         5292 non-null   float64
 2   AMXB       5292 non-null   float64
 3   AXIA3      5292 non-null   float64
 4   BAP        5292 non-null   float64
 5   BBAS3      5292 non-null   float64
 6   BBD        5292 non-null   float64
 7   BIMBOA     5292 non-null   float64
 8   BSAC       5292 non-null   float64
 9   CEMEXCPO   5292 non-null   float64
 10  CENCOSUD   5292 non-null   float64
 11  CIB        5292 non-null   float64
 12  FALABELLA  5292 non-null   float64
 13  FEMSAUBD   5292 non-null   float64
 14  GCARSOA1   5292 non-null   float64
 15  GFNORTEO   5292 non-null   float64
 16  GGB        5292 non-null   float64
 17  GMEXICOB   5292 non-null   float64
 18  ISA        5292 non-null   float64
 19  ITSA4      5292 non-null   float64
 20  ITUB     

In [13]:
##### Analizando empresas excluidas y creando datos de rendimientos logaritmicos
#### Empresas excluidas
not_included = set(tickers).difference(set(df_precios.columns.to_list()))
    # Ojo: "tickers" ya fue procesado de "AMXB.MX" -> "AMXB"
final_tickers = [t for t in tickers if t not in not_included]

#### Creando rendimientos regulares
df_rendis = df_precios.pct_change().dropna()

#### Formula de retornos logaritmicos
    # R_{t} = log(P_{t}/P_{t-1}) = log(P_{t}) - log(P_{t-1})
df_rendis_log = np.log(df_precios) - np.log(df_precios.shift(1))
df_rendis_log = df_rendis_log.drop(index=df_rendis_log.index[0]) #Primera fila es NaN

print(not_included)
display(df_rendis_log.head())
display(df_rendis_log.tail())


{'CHILE', 'LTM', 'NU', 'B3SA3', 'RDOR3', 'USD', 'EC', 'FUNO11'}


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,-0.009202,0.0,-0.008253,0.000000,-0.008981,0.000000,-0.007209,-0.016039,-0.018774,-0.017564,...,-0.011924,-0.013538,0.000000,0.000000,0.014798,-0.002043,-0.022994,0.004263,-0.005655,0.0
2006-03-01,0.026263,0.0,0.028861,0.019894,0.001336,0.029082,0.037866,0.025278,0.010682,0.012556,...,0.051864,0.064523,0.037842,0.033419,0.013716,-0.010282,0.033256,0.023125,0.008305,0.0
2006-03-02,0.011125,0.0,-0.001074,-0.026317,-0.019883,-0.011460,0.001700,0.001050,-0.002085,-0.006258,...,0.005732,0.004679,-0.006921,0.001548,0.038506,0.000000,-0.000834,0.014852,0.010202,0.0
2006-03-03,0.010468,0.0,-0.000537,-0.006689,-0.023417,0.008607,-0.009355,-0.010554,-0.004394,0.008690,...,0.001078,0.003146,0.010892,-0.007441,0.018302,-0.005306,-0.001252,-0.011118,-0.000983,0.0
2006-03-06,-0.013441,0.0,0.021277,-0.036451,-0.008749,-0.018197,-0.043459,-0.027429,-0.029364,-0.008537,...,-0.035973,-0.041685,0.029931,-0.006974,-0.044990,-0.014483,-0.028374,-0.009570,-0.017524,0.0


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2026-06-08,-0.016155,-0.002090,0.001383,-0.004739,-0.008126,-0.003658,-0.021053,0.003936,-0.001333,-0.028026,...,0.000000,0.003150,-0.010155,-0.002564,-0.014500,-0.036039,-0.015884,0.000000,0.006436,0.035627
2026-06-09,0.016155,-0.019721,0.015540,-0.001188,0.088841,0.000523,0.012085,-0.006270,0.046277,-0.000474,...,0.003936,0.000000,0.016786,0.015284,0.027139,0.039083,0.009957,0.000000,-0.016860,-0.015344
2026-06-10,-0.003210,-0.007186,0.003170,0.001386,0.005847,-0.005773,-0.009050,0.020102,-0.003829,-0.025439,...,0.016143,0.011879,-0.043273,-0.006158,-0.043223,-0.018947,-0.013968,0.000780,-0.009136,-0.021933
2026-06-11,0.034759,0.003719,0.077396,0.033855,0.046450,0.021349,0.041549,0.026932,0.035797,0.061282,...,0.007153,0.015418,0.042048,0.011920,0.082351,0.080209,0.028394,0.026935,0.031231,-0.000944
2026-06-12,0.009274,0.001284,0.000836,-0.005179,0.003171,0.002573,0.017291,-0.001544,0.002772,0.019919,...,0.007646,-0.001225,-0.002454,-0.011193,0.041033,0.044733,0.022531,0.014324,0.008473,0.006121


In [14]:
#### Guardando retornos logaritmicos
df_rendis.to_csv('retornos.csv', index=True, encoding='utf-8')
# df_rendis_log.to_csv('retornos_log_splac.csv', index=True, encoding='utf-8')